## Gold Baseline Modeling (Deliverable 1.3.2)

This notebook implements the baseline anomaly detection model using a single, broad Isolation Forest trained on the complete Gold feature set. The baseline serves as the primary comparison point for evaluating whether the three-stage cascade model improves alert quality.

**Purpose:**  
To produce the baseline model’s anomaly scores and alert outputs using the fully preprocessed Gold dataset. These outputs represent the simplest form of unsupervised anomaly detection and form the quantitative reference for the comparative evaluation in the Gold Comparison notebook.

**Key Goals:**

- Load the Gold preprocessed dataset and Stage 1 feature set.
- Train a single Isolation Forest using all vetted numeric features.
- Generate anomaly scores and binary alert flags.
- Produce baseline alert frequency counts, false-positive counts, and normal-period alert rates.
- Save all baseline model artifacts, metrics, and alert outputs for use in the Gold Comparison notebook.

**Relevance to Section C:**  
This notebook provides the baseline alert patterns used in Section C to evaluate whether the cascade reduces false positives, reduces noisy alerts, and preserves meaningful anomaly sensitivity. The outputs here form the “reference condition” for the paired statistical tests and practical significance analysis.

In [98]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

#import os
#import glob

from pathlib import Path
import yaml
import re

import logging
import wandb

import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 

import joblib 

from sklearn.model_selection import train_test_split, KFold

from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, RobustScaler

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, roc_auc_score, average_precision_score

from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor

import pyarrow.parquet as pq
import pyarrow as pa


import hashlib


# Custom Utilities Module
from utils.paths import get_paths
from utils.file_io import load_data, save_data, save_json, load_json
from utils.eda_logging import profile_dataframe
from utils.logging_setup import configure_logging
from utils.wandb_utils import finalize_wandb_stage

# Ledger 
from utils.ledger import Ledger

# Show more columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)


In [99]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [100]:
def log_gold_paths(paths, logger: logging.Logger) -> None:
    logger.info("Project Root Path Loaded: %s", paths.root)
    logger.info("Project Logging Path Loaded: %s", paths.logs)
    logger.info("Project Artifacts Path Loaded: %s", paths.artifacts)
    logger.info("Project Notebooks Path Loaded %s", paths.notebooks)
    logger.info("Project Data Path Loaded: %s", paths.data)
    logger.info("Data Bronze Path Loaded: %s", paths.data_bronze)
    logger.info("Data Bronze Training Path Loaded: %s", paths.data_bronze_train)
    logger.info("Data Bronze Testing Path Loaded: %s", paths.data_bronze_test)
    logger.info("Data Silver Path Loaded: %s", paths.data_silver)
    logger.info("Data Silver Training Path Loaded: %s", paths.data_silver_train)
    logger.info("Data Silver Testing Path Loaded: %s", paths.data_silver_test)
    logger.info("Data Gold Path Loaded: %s", paths.data_gold)

In [101]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [102]:
# Configurables

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Stage Details
STAGE = "gold"
LAYER_NAME = "gold"
GOLD_VERSION = "gold__001"
RECIPE_ID = "gold_modeling__v001_baseline"


DATASET_NAME_CONFIG = "pump"
DATASET_NAME = str(DATASET_NAME_CONFIG).strip().lower()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Weights and Biases
WANDB_PROJECT = "capstone"
WANDB_ENTITY = "dcoo230-western-governors-university"
WANDB_RUN_NAME = f"{GOLD_VERSION}"


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# File names
#GOLD_FILE_NAME = f"{DATASET_NAME}__gold__preprocessed.parquet"

GOLD_FIT_FILE_NAME = f"{DATASET_NAME}__gold__fit_normal_only.parquet"
GOLD_TEST_FILE_NAME = f"{DATASET_NAME}__gold__test.parquet"

STAGE1_FEATURES_FILE_NAME = f"{DATASET_NAME}__gold__stage1_features.json"


BASELINE_RESULTS_FILE_NAME = f"{DATASET_NAME}__gold__baseline_results.csv"
BASELINE_MODEL_FILE_NAME = f"{DATASET_NAME}__gold__baseline_isolation_forest.joblib"

BASELINE_THRESHOLDS_FILE_NAME = f"{DATASET_NAME}__gold__baseline_thresholds.json"
BASELINE_SUMMARY_FILE_NAME = f"{DATASET_NAME}__gold__baseline_summary.json"
BASELINE_METADATA_FILE_NAME = f"{DATASET_NAME}__gold__baseline_metadata.json"


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Split configuration
TRAIN_FRACTION = 0.70
RANDOM_SEED = 42

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Scaling posture
USE_ROBUST_SCALER = False

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Baseline thresholding
BASELINE_THRESHOLD_PERCENTILE = 97.0

# Cascade thresholding
STAGE1_THRESHOLD_PERCENTILE = 95.0   # broader / more sensitive
STAGE2_THRESHOLD_PERCENTILE = 98.5   # narrower / stricter

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Isolation Forest size
BASELINE_ESTIMATOR_COUNT = 200


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

GOLD_BASELINE_LEDGER_FILE_NAME = f"ledger__{DATASET_NAME}__gold_baseline_modeling.json"


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 


In [103]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [104]:
# Paths Setup

# Get Path's Object
paths = get_paths()

# Gold
GOLD_FIT_DATA_PATH = paths.data_gold / GOLD_FIT_FILE_NAME
GOLD_TEST_DATA_PATH = paths.data_gold / GOLD_TEST_FILE_NAME
GOLD_ARTIFACTS_PATH = paths.artifacts / "gold" / DATASET_NAME

MODELS_PATH = paths.models / DATASET_NAME
BASELINE_MODELS_PATH = MODELS_PATH / BASELINE_MODEL_FILE_NAME

# Baseline outputs
BASELINE_RESULTS_PATH = GOLD_ARTIFACTS_PATH / BASELINE_RESULTS_FILE_NAME
BASELINE_MODEL_ARTIFACT_PATH = GOLD_ARTIFACTS_PATH / BASELINE_MODEL_FILE_NAME
BASELINE_THRESHOLDS_PATH = GOLD_ARTIFACTS_PATH / BASELINE_THRESHOLDS_FILE_NAME 
BASELINE_SUMMARY_PATH = GOLD_ARTIFACTS_PATH / BASELINE_SUMMARY_FILE_NAME
BASELINE_METADATA_PATH = GOLD_ARTIFACTS_PATH / BASELINE_METADATA_FILE_NAME

STAGE1_FEATURES_PATH = GOLD_ARTIFACTS_PATH / STAGE1_FEATURES_FILE_NAME

# Logs
LOGS_PATH = paths.logs

GOLD_FIT_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_TEST_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)



In [105]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [106]:
# Logging Setup

# Create gold log path 
gold_log_path = paths.logs / "gold_modeling_baseline.log"

# Initial Logger
configure_logging(
    "capstone",
    gold_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.gold")

# Log load and initiation
logger.info("Gold Modeling stage starting")

# Log paths loads
log_gold_paths(paths, logger)


2026-03-02 02:24:04,299 | INFO | capstone.gold | Gold Modeling stage starting
2026-03-02 02:24:04,301 | INFO | capstone.gold | Project Root Path Loaded: /workspace
2026-03-02 02:24:04,304 | INFO | capstone.gold | Project Logging Path Loaded: /workspace/logs
2026-03-02 02:24:04,305 | INFO | capstone.gold | Project Artifacts Path Loaded: /workspace/artifacts
2026-03-02 02:24:04,307 | INFO | capstone.gold | Project Notebooks Path Loaded /workspace/notebooks
2026-03-02 02:24:04,308 | INFO | capstone.gold | Project Data Path Loaded: /workspace/data
2026-03-02 02:24:04,309 | INFO | capstone.gold | Data Bronze Path Loaded: /workspace/data/bronze
2026-03-02 02:24:04,311 | INFO | capstone.gold | Data Bronze Training Path Loaded: /workspace/data/bronze/train
2026-03-02 02:24:04,312 | INFO | capstone.gold | Data Bronze Testing Path Loaded: /workspace/data/bronze/test
2026-03-02 02:24:04,315 | INFO | capstone.gold | Data Silver Path Loaded: /workspace/data/silver
2026-03-02 02:24:04,317 | INFO | c

In [107]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [108]:
# W&B

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=WANDB_RUN_NAME,
    job_type="gold_modeling_baseline",
    config={
        "gold_version": GOLD_VERSION,
        "dataset": DATASET_NAME,
        "stage": STAGE,
        "train_fraction": TRAIN_FRACTION,
        "baseline_threshold_percentile": BASELINE_THRESHOLD_PERCENTILE,
        "gold_input_path": str(GOLD_FIT_DATA_PATH),
    },
)
logger.info("W&B initialized: %s", wandb.run.name)


2026-03-02 02:24:05,381 | INFO | capstone.gold | W&B initialized: gold__001


In [109]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [110]:
# Ledger Setup

ledger = Ledger(stage=STAGE, recipe_id=RECIPE_ID)

ledger.add(
    kind="step",
    step="init",
    message="Initialized ledger",
    logger=logger
)


2026-03-02 02:24:05,955 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-02T02:24:05.955233+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'init', 'message': 'Initialized ledger', 'why': None, 'consequence': None, 'data': {}}


{'ts_utc': '2026-03-02T02:24:05.955233+00:00',
 'stage': 'gold',
 'recipe': 'gold_modeling__v001_baseline',
 'kind': 'step',
 'step': 'init',
 'message': 'Initialized ledger',
 'why': None,
 'consequence': None,
 'data': {}}

In [111]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [112]:
logger.info("Loading Gold fit parquet: %s", GOLD_FIT_DATA_PATH)
gold_fit_dataframe = load_data(GOLD_FIT_DATA_PATH)

logger.info("Loading Gold test parquet: %s", GOLD_TEST_DATA_PATH)
gold_test_dataframe = load_data(GOLD_TEST_DATA_PATH)

logger.info("Loading Stage 1 features: %s", STAGE1_FEATURES_PATH)
stage1_feature_columns = load_json(STAGE1_FEATURES_PATH)

ledger.add(
    kind="step",
    step="load_modeling_inputs",
    message="Loaded saved Gold fit/test parquet artifacts and Stage 1 feature list.",
    data={
        "gold_fit_path": str(GOLD_FIT_DATA_PATH),
        "gold_test_path": str(GOLD_TEST_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "gold_fit_shape": list(gold_fit_dataframe.shape),
        "gold_test_shape": list(gold_test_dataframe.shape),
        "stage1_feature_count": int(len(stage1_feature_columns)),
    },
    logger=logger,
)

gold_test_dataframe.head(3)

2026-03-02 02:24:06,568 | INFO | capstone.gold | Loading Gold fit parquet: /workspace/data/gold/pump__gold__fit_normal_only.parquet
2026-03-02 02:24:06,572 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__fit_normal_only.parquet
2026-03-02 02:24:07,219 | INFO | capstone.gold | Loading Gold test parquet: /workspace/data/gold/pump__gold__test.parquet
2026-03-02 02:24:07,224 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__test.parquet
2026-03-02 02:24:07,375 | INFO | capstone.gold | Loading Stage 1 features: /workspace/artifacts/gold/pump/pump__gold__stage1_features.json
2026-03-02 02:24:07,383 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/gold/pump/pump__gold__stage1_features.json
2026-03-02 02:24:07,392 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-02T02:24:07.392893+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'load_modeling_inputs', 'message': 'Loaded saved

,meta__asset_id,meta__cleaning_recipe_id,meta__dataset,meta__dataset_name,meta__dataset_source,meta__event_id,meta__event_time_parse_success_percent,meta__event_time_source,meta__feature_count,meta__feature_set_id,meta__has_label_candidates,meta__has_status_candidates,meta__ingested_at_utc,meta__label_source,meta__label_source_column,meta__label_source_kind,meta__label_type,meta__layer,meta__processed_at_utc,meta__record_id,meta__run_id,meta__silver_version,meta__source_file,meta__source_row_id,meta__split,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,sensor_30,sensor_31,sensor_32,sensor_33,sensor_34,sensor_35,sensor_36,sensor_37,sensor_38,sensor_39,sensor_40,sensor_41,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_51,timestamp,machine_status,meta__gold_imputation,meta__gold_scaler
0,asset__001,silver__clean__dataset__agnostic__v001,pump,pump,meta__dataset,pump:asset__001:run__001:176256,100.0,timestamp,50,feature_set__bea5fdd737ae53fcd80ca84cafcd0c40,0,1,2026-03-01 03:54:51.311816+00:00,<NA>,machine_status,status,<NA>,gold,2026-03-01 04:18:23.781532+00:00,5992450321897436818,run__001,silver__001,sensor.csv,176256,unsplit,2018-08-01 09:36:00+00:00,176256,176256,2018-08-01 00:00:00+00:00,0,0,1,NORMAL,2.453588,49.348957,51.51910,44.53125,642.129639,70.64276,14.22888,16.65220,15.14757,15.38628,43.55097,52.65041,34.85802,19.25599,420.7400,465.5606,473.3555,2.672640,668.1029,400.0775,881.1509,532.5089,1092.967,629.4528,744.5596,977.3638,485.1414,941.6569,596.5172,685.6481,961.4583,1011.959,557.5650,319.7289,476.5692,813.5801,28.63851,46.09375,34.37500,75.52083,32.55208,34.11458,44.27083,47.45370,42.24537,49.768520,42.53472,312.5000,66.84028,210.9375,2018-08-01 09:36:00,NORMAL,forward_fill_within_group_then_median,none
1,asset__001,silver__clean__dataset__agnostic__v001,pump,pump,meta__dataset,pump:asset__001:run__001:176257,100.0,timestamp,50,feature_set__bea5fdd737ae53fcd80ca84cafcd0c40,0,1,2026-03-01 03:54:51.311816+00:00,<NA>,machine_status,status,<NA>,gold,2026-03-01 04:18:23.781532+00:00,2078900024144198128,run__001,silver__001,sensor.csv,176257,unsplit,2018-08-01 09:37:00+00:00,176257,176257,2018-08-01 00:00:00+00:00,0,0,1,NORMAL,2.453588,49.348960,51.60590,44.53125,630.786987,71.26656,14.39525,16.75347,15.18374,15.16204,45.64900,52.03297,34.67496,19.94810,420.0238,462.5600,465.0322,2.573149,665.9155,399.6319,879.0721,534.0840,1093.140,626.4788,739.2676,980.0061,496.7717,941.4617,539.0922,716.2037,952.6041,1009.355,558.2068,325.1864,492.9296,813.3455,30.54268,45.83333,34.37500,83.07291,32.55208,33.85416,44.01041,47.74306,41.95602,49.768517,43.11343,318.2870,66.55093,209.4907,2018-08-01 09:37:00,NORMAL,forward_fill_within_group_then_median,none
2,asset__001,silver__clean__dataset__agnostic__v001,pump,pump,meta__dataset,pump:asset__001:run__001:176258,100.0,timestamp,50,feature_set__bea5fdd737ae53fcd80ca84cafcd0c40,0,1,2026-03-01 03:54:51.311816+00:00,<NA>,machine_status,status,<NA>,gold,2026-03-01 04:18:23.781532+00:00,5178773400662276631,run__001,silver__001,sensor.csv,176258,unsplit,2018-08-01 09:38:00+00:00,176258,176258,2018-08-01 00:00:00+00:00,0,0,1,NORMAL,2.451620,49.305550,51.47569,44.57465,634.606445,70.15800,14.43866,16.78964,15.18374,15.11140,45.16984,51.57830,34.54854,19.92392,421.2197,463.6213,462.7087,2.522767,666.9105,398.2779,882.9197,534.3575,1092.486,630.1271,742.2440,979.2006,507.1257,953.0093,552.8891,701.3889,995.3124,1021.922,567.2665,317.8607,486.9266,804.4023,41.09998,45.83333,35.41666,90.36458,32.29166,33.85416,45.57291,47.45370,41.66667,49.768520,43.11343,320.0231,66.26157,206.5972,2018-08-01 09:38:00,NORMAL,forward_fill_withi

In [113]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [114]:
def build_reference_profile(
    dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
) -> pd.DataFrame:
    reference_rows: list[dict] = []

    for feature_name in feature_columns:
        feature_series = dataframe[feature_name]

        reference_rows.append({
            "feature_name": feature_name,
            "median_value": float(feature_series.median()),
            "mean_value": float(feature_series.mean()),
            "standard_deviation": float(feature_series.std()) if not pd.isna(feature_series.std()) else 0.0,
            "lower_bound": float(feature_series.quantile(0.05)),
            "upper_bound": float(feature_series.quantile(0.95)),
        })

    reference_profile = pd.DataFrame(reference_rows)
    return reference_profile

In [115]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [116]:
baseline_train_fit_features = gold_fit_dataframe[stage1_feature_columns].values
baseline_test_features = gold_test_dataframe[stage1_feature_columns].values

if "anomaly_flag" in gold_test_dataframe.columns:
    test_labels = gold_test_dataframe["anomaly_flag"].astype(int).values
else:
    test_labels = None

In [117]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [118]:
def compute_anomaly_scores_isolation_forest(
    model: IsolationForest,
    feature_matrix: np.ndarray,
) -> np.ndarray:
    scores = model.score_samples(feature_matrix)
    anomaly_scores = -scores
    return anomaly_scores

In [119]:
def choose_threshold_by_percentile(
    anomaly_scores: np.ndarray,
    percentile: float,
) -> float:
    return float(np.percentile(anomaly_scores, percentile))

In [120]:

def evaluate_against_labels(
    true_labels: np.ndarray,
    anomaly_scores: np.ndarray,
    threshold: float,
) -> dict:
    predicted_labels = (anomaly_scores >= threshold).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        predicted_labels,
        average="binary",
        zero_division=0,
    )

    results = {
        "threshold": float(threshold),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }

    if len(np.unique(true_labels)) == 2:
        results["roc_auc"] = float(roc_auc_score(true_labels, anomaly_scores))
        results["pr_auc"] = float(average_precision_score(true_labels, anomaly_scores))
    else:
        results["roc_auc"] = None
        results["pr_auc"] = None

    return results

In [121]:
baseline_model = IsolationForest(
    n_estimators=BASELINE_ESTIMATOR_COUNT,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

baseline_model.fit(baseline_train_fit_features)

baseline_train_scores = compute_anomaly_scores_isolation_forest(
    baseline_model,
    baseline_train_fit_features,
)

baseline_test_scores = compute_anomaly_scores_isolation_forest(
    baseline_model,
    baseline_test_features,
)

baseline_threshold = choose_threshold_by_percentile(
    baseline_train_scores,
    BASELINE_THRESHOLD_PERCENTILE,
)

baseline_flags = (baseline_test_scores >= baseline_threshold).astype(int)

baseline_results = gold_test_dataframe.copy()
baseline_results["baseline_score"] = baseline_test_scores
baseline_results["baseline_flag"] = baseline_flags

baseline_metrics = {
    "model": "Baseline IsolationForest",
    "threshold_percentile": BASELINE_THRESHOLD_PERCENTILE,
    "threshold": float(baseline_threshold),
    "alert_count_all_rows": int(baseline_flags.sum()),
    "alert_count_test_rows": int(baseline_flags.sum()),
}

if test_labels is not None:
    baseline_metrics.update(
        evaluate_against_labels(
            test_labels,
            baseline_test_scores,
            baseline_threshold,
        )
    )

ledger.add(
    kind="step",
    step="run_baseline_isolation_forest",
    message="Ran baseline Isolation Forest using saved Gold fit data and scored the saved Gold test data.",
    data={
        "estimator_count": int(BASELINE_ESTIMATOR_COUNT),
        "threshold_percentile": float(BASELINE_THRESHOLD_PERCENTILE),
        "threshold": float(baseline_threshold),
        "training_rows": int(len(gold_fit_dataframe)),
        "test_rows": int(len(gold_test_dataframe)),
        "feature_count": int(len(stage1_feature_columns)),
        "alert_count_test_rows": int(baseline_flags.sum()),
        "precision": baseline_metrics.get("precision"),
        "recall": baseline_metrics.get("recall"),
        "f1": baseline_metrics.get("f1"),
        "roc_auc": baseline_metrics.get("roc_auc"),
        "pr_auc": baseline_metrics.get("pr_auc"),
    },
    logger=logger,
)

baseline_metrics

2026-03-02 02:24:12,113 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-02T02:24:12.113371+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'run_baseline_isolation_forest', 'message': 'Ran baseline Isolation Forest using saved Gold fit data and scored the saved Gold test data.', 'why': None, 'consequence': None, 'data': {'estimator_count': 200, 'threshold_percentile': 97.0, 'threshold': 0.5507766554570627, 'training_rows': 161772, 'test_rows': 44064, 'feature_count': 50, 'alert_count_test_rows': 594, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'roc_auc': None, 'pr_auc': None}}


{'model': 'Baseline IsolationForest',
 'threshold_percentile': 97.0,
 'threshold': 0.5507766554570627,
 'alert_count_all_rows': 594,
 'alert_count_test_rows': 594,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': None,
 'pr_auc': None}

In [122]:
baseline_alert_count_all_rows = int(baseline_results["baseline_flag"].sum())
baseline_alert_count_test_rows = int(baseline_results["baseline_flag"].sum())

baseline_thresholds = {
    "baseline_threshold_percentile": float(BASELINE_THRESHOLD_PERCENTILE),
    "baseline_threshold": float(baseline_threshold),
}

baseline_summary = {
    "dataset_name": DATASET_NAME,
    "baseline_metrics": baseline_metrics,
    "alert_count_all_rows": baseline_alert_count_all_rows,
    "alert_count_test_rows": baseline_alert_count_test_rows,
    "result_row_count": int(len(baseline_results)),
}

baseline_metadata = {
    "gold_fit_path": str(GOLD_FIT_DATA_PATH),
    "gold_test_path": str(GOLD_TEST_DATA_PATH),
    "baseline_results_path": str(BASELINE_RESULTS_PATH),
    "baseline_model_path": str(BASELINE_MODELS_PATH),
    "baseline_model_path": str(BASELINE_MODEL_ARTIFACT_PATH),
    "baseline_thresholds_path": str(BASELINE_THRESHOLDS_PATH),
    "baseline_summary_path": str(BASELINE_SUMMARY_PATH),
}

baseline_results.to_csv(BASELINE_RESULTS_PATH, index=False)
joblib.dump(baseline_model, BASELINE_MODEL_ARTIFACT_PATH)
joblib.dump(baseline_model, BASELINE_MODELS_PATH)

save_json(baseline_thresholds, BASELINE_THRESHOLDS_PATH)
save_json(baseline_summary, BASELINE_SUMMARY_PATH)
save_json(baseline_metadata, BASELINE_METADATA_PATH)

wandb.save(str(BASELINE_RESULTS_PATH))
wandb.save(str(BASELINE_MODEL_ARTIFACT_PATH))
wandb.save(str(BASELINE_MODELS_PATH))
wandb.save(str(BASELINE_THRESHOLDS_PATH))
wandb.save(str(BASELINE_SUMMARY_PATH))
wandb.save(str(BASELINE_METADATA_PATH))

ledger.add(
    kind="step",
    step="save_baseline_outputs",
    message="Saved baseline results, trained Isolation Forest model, thresholds, summary, and metadata.",
    data={
        "baseline_results_path": str(BASELINE_RESULTS_PATH),
        "baseline_model_artifact_path": str(BASELINE_MODEL_ARTIFACT_PATH),
        "baseline_model_path": str(BASELINE_MODELS_PATH),
        "baseline_thresholds_path": str(BASELINE_THRESHOLDS_PATH),
        "baseline_summary_path": str(BASELINE_SUMMARY_PATH),
        "baseline_metadata_path": str(BASELINE_METADATA_PATH),
        "result_row_count": int(len(baseline_results)),
        "alert_count_all_rows": baseline_alert_count_all_rows,
        "alert_count_test_rows": baseline_alert_count_test_rows,
    },
    logger=logger,
)

2026-03-02 02:24:19,834 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__baseline_thresholds.json
2026-03-02 02:24:19,850 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__baseline_summary.json
2026-03-02 02:24:19,865 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__baseline_metadata.json
2026-03-02 02:24:20,108 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-02T02:24:20.108673+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'save_baseline_outputs', 'message': 'Saved baseline results, trained Isolation Forest model, thresholds, summary, and metadata.', 'why': None, 'consequence': None, 'data': {'baseline_results_path': '/workspace/artifacts/gold/pump/pump__gold__baseline_results.csv', 'baseline_model_artifact_path': '/workspace/artifacts/gold/pump/pump__gold__baseline_isolation_forest.joblib', 'baseline_model_path': '/workspace/models/pum

{'ts_utc': '2026-03-02T02:24:20.108673+00:00',
 'stage': 'gold',
 'recipe': 'gold_modeling__v001_baseline',
 'kind': 'step',
 'step': 'save_baseline_outputs',
 'message': 'Saved baseline results, trained Isolation Forest model, thresholds, summary, and metadata.',
 'why': None,
 'consequence': None,
 'data': {'baseline_results_path': '/workspace/artifacts/gold/pump/pump__gold__baseline_results.csv',
  'baseline_model_artifact_path': '/workspace/artifacts/gold/pump/pump__gold__baseline_isolation_forest.joblib',
  'baseline_model_path': '/workspace/models/pump/pump__gold__baseline_isolation_forest.joblib',
  'baseline_thresholds_path': '/workspace/artifacts/gold/pump/pump__gold__baseline_thresholds.json',
  'baseline_summary_path': '/workspace/artifacts/gold/pump/pump__gold__baseline_summary.json',
  'baseline_metadata_path': '/workspace/artifacts/gold/pump/pump__gold__baseline_metadata.json',
  'result_row_count': 44064,
  'alert_count_all_rows': 594,
  'alert_count_test_rows': 594}}

In [123]:
ledger.add(
    kind="step",
    step="finalize_baseline_modeling",
    message="Gold baseline modeling notebook complete.",
    data={
        "baseline_metrics": baseline_metrics,
        "baseline_results_path": str(BASELINE_RESULTS_PATH),
        "baseline_model_artifact_path": str(BASELINE_MODEL_ARTIFACT_PATH),
        "baseline_model_path": str(BASELINE_MODELS_PATH),
    },
    logger=logger,
)

baseline_ledger_path = GOLD_ARTIFACTS_PATH / GOLD_BASELINE_LEDGER_FILE_NAME
ledger.write_json(baseline_ledger_path)

wandb.save(str(baseline_ledger_path))
wandb_run.finish()

2026-03-02 02:24:20,415 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-02T02:24:20.415783+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'finalize_baseline_modeling', 'message': 'Gold baseline modeling notebook complete.', 'why': None, 'consequence': None, 'data': {'baseline_metrics': {'model': 'Baseline IsolationForest', 'threshold_percentile': 97.0, 'threshold': 0.5507766554570627, 'alert_count_all_rows': 594, 'alert_count_test_rows': 594, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'roc_auc': None, 'pr_auc': None}, 'baseline_results_path': '/workspace/artifacts/gold/pump/pump__gold__baseline_results.csv', 'baseline_model_artifact_path': '/workspace/artifacts/gold/pump/pump__gold__baseline_isolation_forest.joblib', 'baseline_model_path': '/workspace/models/pump/pump__gold__baseline_isolation_forest.joblib'}}
